[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S07/2_Practica/para_casa/S07_03_regresion_logistica.ipynb)

# S07 · 03 — Regresión logística: predecir cancelaciones

Sprint 7 · M3 (UD 3.4)

**Contexto TurisData.** El equipo de reservas quiere saber **qué reservas tienen riesgo de cancelarse** para actuar a tiempo (pedir un depósito, hacer un recordatorio, vigilar el *overbooking*). Es un problema de **clasificación** (0 = se mantiene, 1 = cancela). La regresión logística es el punto de partida: rápida, explicable y da **probabilidades**.

**Este cuaderno tiene 4 ejercicios y un mini-reto.** Los ejercicios 1, 2 y 3 son los mismos que los del cuaderno «en clase» (allí, del 1 al 3).

**Cómo usar este cuaderno** (tiene dos partes):

- **Parte A · Ejecutar** (aquí abajo): de arriba abajo, cada celda de código lleva encima su título con un símbolo:
    - ⚙️ **Preparación**: se ejecuta sola, pulsa ▶ y sigue.
    - 👀 **Ejemplo**: pulsa ▶, mira el resultado y responde las preguntas 🧠 (no hay que escribir código). Las respuestas están en la Parte B.
    - ✏️ **Ejercicio**: tu turno. Cambia los `...` y pulsa ▶. Si sale **✔**, responde las preguntas 🧠 y sigues. Para comprobar tu código, tienes la solución de referencia ✅ en la Parte B.
- **Parte B · Explicación** (al final): para cada celda, **qué hace**, **qué resultado debe salir** y, en los ejemplos, las **respuestas** a las preguntas 🧠 y, en los ejercicios, la **solución de referencia** ✅ para comprobar tu código. Ábrela cuando quieras entender una celda, comprobar tus respuestas o no te salga. Usa el índice de la izquierda (☰) para saltar.

# Parte A · Ejecutar

### ⚙️ A1 · PREPARACIÓN — se ejecuta sola

Pulsa ▶ y sigue. No hace falta leerla.

In [ ]:
#@title A1 · ▶ Ejecuta esta celda primero (no hace falta leerla)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos", "../../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_predict, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, precision_score, recall_score, accuracy_score

SEMILLA = 42
sns.set_theme(style="whitegrid")

df = pd.read_csv(dato("reservas_turisdata.csv"), parse_dates=["fecha_reserva", "fecha_llegada"])
df["mes_llegada"] = df["fecha_llegada"].dt.month
df["temporada_alta"] = df["mes_llegada"].isin([1, 2, 3, 7, 8, 12]).astype(int)

columnas_num = ["antelacion_dias", "noches", "adultos", "ninos", "precio_noche",
                "peticiones_especiales", "cliente_repetidor", "cancelaciones_previas", "temporada_alta"]
columnas_cat = ["tipo_habitacion", "regimen", "canal", "pais_origen", "tarifa", "deposito"]
X, y = df[columnas_num + columnas_cat], df["cancelada"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEMILLA, stratify=y)
print(f"Tasa de cancelación: {y.mean():.1%} · train {len(X_train)} · test {len(X_test)}")
print("Listo ✔")

### 👀 A2 · EJEMPLO — La función sigmoide

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A2).

1. ¿Qué probabilidad corresponde a `z = 0`?
2. ¿Qué le pasa a la probabilidad cuando `z` es muy grande o muy negativo?
3. ¿Por qué hace falta la sigmoide?

In [ ]:
z = np.linspace(-6, 6, 200)
plt.figure(figsize=(6, 3.5))
plt.plot(z, 1 / (1 + np.exp(-z)), color="#2a6f97", lw=2)
plt.axhline(0.5, color="gray", ls="--")
plt.title("Función sigmoide: convierte cualquier número en una probabilidad")
plt.xlabel("Suma ponderada z")
plt.ylabel("Probabilidad")
plt.show()

### ✏️ A3 · EJERCICIO 1 de 4 — Entrenar el modelo y obtener probabilidades

**Tu turno:** Crea `logit`, un `Pipeline` con el preprocesado que ves abajo (`prep`) y `LogisticRegression(max_iter=1000)`. Ajústalo con train y guarda en `proba_test` la probabilidad de cancelar (clase 1) de cada reserva de test.

**Debe salir:** ✔ Ejercicio 1 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A3).

1. ¿Cuántas probabilidades hay en `proba_test` y entre qué valores están?
2. ¿Qué probabilidad media sale y cómo se compara con la tasa de cancelación (45,9 %)?
3. ¿Qué diferencia hay entre `predict` y `predict_proba(X)[:, 1]`?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A3, **después de intentarlo**.

In [ ]:
prep = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("esc", StandardScaler())]), columnas_num),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("oh", OneHotEncoder(handle_unknown="ignore", drop="if_binary"))]), columnas_cat),
])
# pista: Pipeline([("prep", prep), ("clf", LogisticRegression(...))]); predict_proba(X_test)[:, 1]
logit = ...
proba_test = ...

assert proba_test.shape == (len(X_test),), "Debe haber una probabilidad por reserva de test"
assert 0 <= proba_test.min() and proba_test.max() <= 1, "Son probabilidades: entre 0 y 1"
assert 0.35 < proba_test.mean() < 0.55, "La probabilidad media debería rondar la tasa de cancelación (≈0,46)"
print("✔ Ejercicio 1 correcto")

### 👀 A4 · EJEMPLO — Odds ratio de cada variable

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A4).

1. ¿Qué color tienen las barras que aumentan el riesgo y cuáles son las tres mayores?
2. ¿Qué tres variables reducen más el riesgo?
3. ¿Cómo se lee el odds ratio de `antelacion_dias` (≈ 2,12)?

In [ ]:
nombres = logit[:-1].get_feature_names_out()
coefs = pd.Series(logit[-1].coef_[0], index=nombres)
odds = np.exp(coefs).sort_values(ascending=False)
plt.figure(figsize=(7, 6))
odds.iloc[::-1].plot(kind="barh", color=["#e63946" if v > 1 else "#2a6f97" for v in odds.iloc[::-1]])
plt.axvline(1, color="black", lw=1)
plt.title("Odds ratio de cada variable (rojo = más riesgo de cancelar)")
plt.xlabel("Odds ratio")
plt.ylabel("")
plt.tight_layout()
plt.show()

### ✏️ A5 · EJERCICIO 2 de 4 — Leer los odds ratio

**Tu turno:** A partir de `odds` (un `pd.Series` cuyo índice son los nombres de las variables), guarda en `or_no_reembolsable` el odds ratio de `cat__tarifa_no_reembolsable` y en `or_deposito_total` el de `cat__deposito_total`. Guarda en `mas_riesgo` el nombre de la variable con **mayor** odds ratio.

**Debe salir:** `Tarifa no reembolsable: los odds de cancelar se multiplican por 0.44`, `Depósito total: los odds de cancelar se multiplican por 0.46` y ✔ Ejercicio 2 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A5).

1. ¿Cuánto valen `or_no_reembolsable` y `or_deposito_total`?
2. Interpreta el 0,44 de la tarifa no reembolsable con tus palabras.
3. ¿Qué variable aumenta más el riesgo y cuánto?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A5, **después de intentarlo**.

In [ ]:
# pista: odds["nombre_de_la_variable"]; idxmax() para el mayor
or_no_reembolsable = ...
or_deposito_total = ...
mas_riesgo = ...

assert or_no_reembolsable < 1, "La tarifa no reembolsable reduce el riesgo de cancelar (odds ratio < 1)"
assert or_deposito_total < 1, "Un depósito total reduce el riesgo de cancelar (odds ratio < 1)"
assert isinstance(mas_riesgo, str) and odds[mas_riesgo] > 1
print(f"Tarifa no reembolsable: los odds de cancelar se multiplican por {or_no_reembolsable:.2f}")
print(f"Depósito total: los odds de cancelar se multiplican por {or_deposito_total:.2f}")
print("✔ Ejercicio 2 correcto")

### 👀 A6 · EJEMPLO — Matriz de confusión con dos umbrales

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A6).

1. Con umbral 0,5, ¿cuántas cancelaciones no prevé el modelo (FN) y cuántas falsas alarmas da (FP)?
2. ¿Y con umbral 0,3?
3. ¿Qué ocurre al bajar el umbral de 0,5 a 0,3?

In [ ]:
def metricas_umbral(y_real, proba, umbral):
    """Matriz de confusión y métricas para un umbral dado."""
    pred = (proba >= umbral).astype(int)
    vn, fp, fn, vp = confusion_matrix(y_real, pred).ravel()
    return {"umbral": umbral, "VN": vn, "FP": fp, "FN": fn, "VP": vp,
            "precision": vp / (vp + fp) if (vp + fp) else np.nan,
            "recall": vp / (vp + fn)}

fig, ejes = plt.subplots(1, 2, figsize=(10, 4))
for ax, u in zip(ejes, [0.5, 0.3]):
    ConfusionMatrixDisplay.from_predictions(y_test, (proba_test >= u).astype(int), ax=ax,
                                            display_labels=["Se mantiene", "Cancela"], cmap="Blues", colorbar=False)
    ax.set_title(f"Umbral {u}")
    ax.set_xlabel("Predicho")
    ax.set_ylabel("Real")
plt.tight_layout()
plt.show()

### ✏️ A7 · EJERCICIO 3 de 4 — Efecto de bajar el umbral

**Tu turno:** Usando `metricas_umbral`, calcula sobre test (`y_test`, `proba_test`) las métricas con umbral 0,5 (`m05`) y con umbral 0,3 (`m03`). Guarda en `sube_recall` la diferencia de recall (`m03` − `m05`) y en `baja_precision` la de precisión (`m05` − `m03`).

**Debe salir:** `Con umbral 0,3 detectas 26 puntos más de las cancelaciones, pero tu precisión baja 10 puntos.` y ✔ Ejercicio 3 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A7).

1. ¿Cuánto valen recall y precisión con umbral 0,5 y con 0,3?
2. ¿Qué significa «detectas 26 puntos más»?
3. ¿Qué se paga a cambio?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A7, **después de intentarlo**.

In [ ]:
# pista: llama a metricas_umbral dos veces (umbral 0.5 y 0.3) y resta los campos que necesites
m05 = ...
m03 = ...
sube_recall = ...
baja_precision = ...

assert m05["VN"] + m05["FP"] + m05["FN"] + m05["VP"] == len(y_test), "La matriz debe sumar todas las reservas de test"
assert sube_recall > 0, "Bajar el umbral debe subir el recall"
assert baja_precision > 0, "Bajar el umbral suele bajar la precisión"
print(f"Con umbral 0,3 detectas {sube_recall*100:.0f} puntos más de las cancelaciones, pero tu precisión baja {baja_precision*100:.0f} puntos.")
print("✔ Ejercicio 3 correcto")

### 👀 A8 · EJEMPLO — Probabilidades de validación cruzada y costes de los errores

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A8).

1. ¿Cuántas probabilidades hay en `proba_cv` y por qué son tantas?
2. ¿Por qué `proba_cv` y no `proba_test` para elegir el umbral?
3. Con FN = 3 y FP = 1, ¿qué umbral predice la regla teórica `coste_FP / (coste_FP + coste_FN)`?

In [ ]:
cv = StratifiedKFold(5, shuffle=True, random_state=SEMILLA)
proba_cv = cross_val_predict(logit, X_train, y_train, cv=cv, method="predict_proba")[:, 1]

# Lo que cuesta, en unidades relativas, cada error. Son decisiones de negocio: no tienen valor por defecto.
COSTE_FN, COSTE_FP = 3.0, 1.0   # FN = cancelación que no detectamos · FP = falsa alarma

def coste_total(y_real, proba, umbral, coste_fn, coste_fp):
    m = metricas_umbral(y_real, proba, umbral)
    return m["FN"] * coste_fn + m["FP"] * coste_fp
print("Probabilidades de validación cruzada:", proba_cv.shape, "· coste FN:", COSTE_FN, "· coste FP:", COSTE_FP)

### ✏️ A9 · EJERCICIO 4 de 4 — Elegir el umbral de menor coste

**Tu turno:** Prueba los umbrales `umbrales = np.arange(0.05, 0.96, 0.05)` sobre `proba_cv` / `y_train` con `coste_total` (pasándole `COSTE_FN` y `COSTE_FP`) y guarda en `mejor_umbral` el que da **menor coste**. Después calcula en `coste_test` el coste de ese umbral sobre test.

**Debe salir:** `Umbral elegido: 0.25 · coste en test: 577 (con 0,5: 813)` y ✔ Ejercicio 4 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A9).

1. ¿Qué umbral sale y cuánto cuesta en test frente al 0,5?
2. ¿Cuánto se ahorra respecto al umbral por defecto?
3. ¿Por qué el umbral óptimo es bajo (0,25) y no 0,5?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A9, **después de intentarlo**.

In [ ]:
umbrales = np.arange(0.05, 0.96, 0.05)
# pista: lista de costes con un for (o una lista por comprensión); np.argmin te da la posición del menor
costes = ...
mejor_umbral = ...
coste_test = ...

assert 0.1 <= mejor_umbral <= 0.4, "Con FN 3 veces más caro que FP, el umbral óptimo debería ser bajo (≈0,25)"
assert coste_test < coste_total(y_test, proba_test, 0.5, COSTE_FN, COSTE_FP), "El umbral elegido debería costar menos que el 0,5 por defecto"
print(f"Umbral elegido: {mejor_umbral:.2f} · coste en test: {coste_test:.0f} (con 0,5: {coste_total(y_test, proba_test, 0.5, COSTE_FN, COSTE_FP):.0f})")
print("✔ Ejercicio 4 correcto")

### 👀 A10 · EJEMPLO — Curva del coste según el umbral

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A10).

1. ¿Dónde está el mínimo de la curva y cuánto vale aproximadamente?
2. ¿Por qué el coste sube tanto hacia la derecha?
3. ¿Qué pasa si se cambia a un FN que cuesta lo mismo que un FP?

In [ ]:
plt.figure(figsize=(7, 3.8))
plt.plot(umbrales, costes, marker="o", color="#2a6f97")
plt.axvline(mejor_umbral, color="#e63946", ls="--", label=f"Mínimo: {mejor_umbral:.2f}")
plt.title("Coste total según el umbral (FN cuesta 3 veces más que FP)")
plt.xlabel("Umbral de decisión")
plt.ylabel("Coste (unidades relativas)")
plt.legend()
plt.show()

### ✏️ A11 · MINI-RETO — Tabla de umbrales para el gerente

**Tu turno:** Construye `tabla_umbrales`, un DataFrame con **tres filas** (umbrales 0,5, 0,3 y el `mejor_umbral`) y las columnas `umbral`, `recall`, `precision` y `coste` medidas en **test**. Después escribe en `recomendacion` (texto) qué umbral propondrías y por qué, con al menos una cifra.

**Debe salir:** ✔ Mini-reto correcto (y la tabla `tabla_umbrales`)

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A11).

1. ¿Qué umbral tiene el menor coste y cuál es su recall y su precisión?
2. Explica con una frase para el gerente qué significa precisión 0,533.
3. ¿Por qué conviene enseñar al gerente recall, precisión y coste, y no solo el coste?

In [ ]:
# pista: una lista de umbrales, metricas_umbral y coste_total en cada uno; luego pd.DataFrame(filas)
tabla_umbrales = ...
recomendacion = ...

assert tabla_umbrales.shape[0] == 3, "Debe haber tres filas"
assert {"umbral", "recall", "precision", "coste"} <= set(tabla_umbrales.columns)
assert isinstance(recomendacion, str) and any(c.isdigit() for c in recomendacion), "La recomendación debe ser texto con al menos una cifra"
display(tabla_umbrales)
print("✔ Mini-reto correcto")

# Parte B · Explicación de cada celda y de cada resultado

Cada apartado lleva el **mismo número** que su celda de la Parte A (A1, A2…). Si una celda no te sale, mira su «Si falla».

#### A1 · Preparación — Preparar las herramientas y los datos

- **Qué hace:**
    - Carga las herramientas: `pandas`, `numpy`, `matplotlib`, `seaborn` y, de `scikit-learn`, la regresión logística, el preprocesado, la validación cruzada y las métricas de clasificación.
    - `dato(...)` busca un archivo de datos del curso y, si no está, lo descarga del repositorio.
    - Lee `reservas_turisdata.csv`, añade `mes_llegada` y `temporada_alta`, separa `X` (variables) e `y` (`cancelada`) y reparte entrenamiento y prueba (80 % / 20 %, estratificado).
- **Qué debe salir:**
    - `Tasa de cancelación: 45.9% · train 4800 · test 1200` y `Listo ✔`.
- **Cómo leerlo:**
    - No hay que entender esta celda: solo ejecutarla **la primera**, y otra vez si reinicias el entorno.
- **Si falla:**
    - `NameError: name 'pd' is not defined` → no has ejecutado esta celda. Ejecútala y repite la que falló.

## 1. La idea: de una suma ponderada a una probabilidad

#### A2 · Ejemplo — La función sigmoide

La regresión logística calcula, igual que la lineal, una **suma ponderada** `z` de las variables. Pero en lugar de devolver `z` como respuesta, lo pasa por la **función sigmoide**, que lo comprime entre 0 y 1: `probabilidad = 1 / (1 + e^(−z))`. Así el resultado se puede leer como **la probabilidad de cancelar**.

- **Qué hace:**
    - `np.linspace(-6, 6, 200)` crea 200 valores de `z` entre −6 y 6.
    - `1 / (1 + np.exp(-z))` aplica la sigmoide a cada uno y `plt.plot` dibuja la curva; la línea de puntos gris marca la probabilidad 0,5.
- **Qué debe salir:**
    - Una curva en forma de «S»: casi 0 a la izquierda, 0,5 en el centro (`z = 0`) y casi 1 a la derecha.
- **Si falla:**
    - `NameError: name 'np' is not defined` → falta ejecutar la celda de preparación.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué probabilidad corresponde a `z = 0`?* → **0,5**: la curva corta la línea gris justo en el centro.
2. *¿Qué le pasa a la probabilidad cuando `z` es muy grande o muy negativo?* → Se acerca a **1** o a **0**, pero nunca llega: siempre queda algo de incertidumbre.
3. *¿Por qué hace falta la sigmoide?* → Porque una suma ponderada puede dar cualquier número (−40, 7, 250…) y una **probabilidad** tiene que estar entre 0 y 1.

</details>

## 2. Entrenar el modelo

#### A3 · Ejercicio 1 — Entrenar el modelo y obtener probabilidades

Como en el Sprint 6, todo el preprocesado va dentro de un `Pipeline` (imputar, escalar y codificar). El modelo devuelve `predict_proba` (probabilidad por clase) y `predict` (la clase, usando un **umbral de 0,5** por defecto).

- **Qué hace:**
    - `prep` es el preprocesado: en las numéricas imputa con la mediana y escala; en las categóricas imputa la más frecuente y las pasa a columnas 0/1 (`drop="if_binary"` deja una sola columna si la variable tiene dos valores).
    - `Pipeline([("prep", prep), ("clf", LogisticRegression(max_iter=1000))]).fit(X_train, y_train)` ajusta todo con train.
    - `logit.predict_proba(X_test)` devuelve dos columnas (probabilidad de no cancelar y de cancelar); `[:, 1]` se queda con la de **cancelar** (clase 1).
- **Qué debe salir:**
    - `✔ Ejercicio 1 correcto`. `proba_test` tiene 1200 valores, de 0,03 a 0,99, con media ≈ 0,466.
- **Si falla:**
    - `Debe haber una probabilidad por reserva de test` → `proba_test` no es `predict_proba(X_test)[:, 1]` (¿falta `[:, 1]` o has usado `X_train`?).
    - `Son probabilidades: entre 0 y 1` → has usado `predict` o `decision_function`, que no son probabilidades.
    - `La probabilidad media debería rondar la tasa de cancelación` → has cogido la columna 0 (probabilidad de no cancelar).
    - `NameError: name 'columnas_num' is not defined` → falta ejecutar la celda de preparación.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuántas probabilidades hay en `proba_test` y entre qué valores están?* → **1200**, una por reserva de test, entre **0,03 y 0,99**.
2. *¿Qué probabilidad media sale y cómo se compara con la tasa de cancelación (45,9 %)?* → **≈ 0,466**, casi igual: en promedio, el modelo predice una probabilidad coherente con lo que ocurre en realidad.
3. *¿Qué diferencia hay entre `predict` y `predict_proba(X)[:, 1]`?* → `predict` da la **clase** (0 o 1, con umbral 0,5); `predict_proba(X)[:, 1]` da la **probabilidad de cancelar** (columna 1 = clase 1), con la que luego podemos elegir otro umbral.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
prep = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("esc", StandardScaler())]), columnas_num),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("oh", OneHotEncoder(handle_unknown="ignore", drop="if_binary"))]), columnas_cat),
])
logit = Pipeline([("prep", prep), ("clf", LogisticRegression(max_iter=1000))]).fit(X_train, y_train)
proba_test = logit.predict_proba(X_test)[:, 1]
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

## 3. Interpretar: coeficientes y *odds ratios*

#### A4 · Ejemplo — Odds ratio de cada variable

Los *odds* son «probabilidad de cancelar / probabilidad de no cancelar». Al hacer `exp(coeficiente)` obtenemos el ***odds ratio***: cuánto se **multiplican los odds** al subir la variable una unidad (en las numéricas, una desviación típica por el escalado) o al pasar a esa categoría.
- *Odds ratio* > 1 → **aumenta** el riesgo. < 1 → lo **reduce**. = 1 → no influye.

- **Qué hace:**
    - `logit[:-1].get_feature_names_out()` da el nombre de cada columna que ve el modelo; `logit[-1].coef_[0]` son sus coeficientes (los une `coefs`).
    - `np.exp(coefs)` los convierte en *odds ratios*; `sort_values(ascending=False)` los ordena de mayor a menor (`odds`).
    - El gráfico de barras los dibuja: **rojo** si es mayor que 1 (más riesgo) y **azul** si es menor; la línea negra marca el 1.
- **Qué debe salir:**
    - Un gráfico de 33 barras horizontales. Arriba (más riesgo): `num__antelacion_dias` (2,12), `cat__deposito_sin_deposito` (2,00) y `cat__canal_web_intermediaria` (1,70). Abajo (menos riesgo): `cat__tarifa_no_reembolsable` (0,44), `cat__deposito_total` (0,46) y `cat__canal_directo` (0,55).
- **Si falla:**
    - `NameError: name 'logit' is not defined` → falta el ejercicio 1 (o su solución en la Parte B).

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué color tienen las barras que aumentan el riesgo y cuáles son las tres mayores?* → **Rojo**. Las mayores: `antelacion_dias` (≈ 2,12), `deposito_sin_deposito` (≈ 2,00) y `canal_web_intermediaria` (≈ 1,70).
2. *¿Qué tres variables reducen más el riesgo?* → `tarifa_no_reembolsable` (≈ 0,44), `deposito_total` (≈ 0,46) y `canal_directo` (≈ 0,55).
3. *¿Cómo se lee el odds ratio de `antelacion_dias` (≈ 2,12)?* → Por cada **desviación típica** más de antelación (las numéricas están escaladas), los odds de cancelar se **multiplican por 2,12**: reservar con mucha antelación se cancela más.

</details>

#### A5 · Ejercicio 2 — Leer los odds ratio

- **Qué hace:**
    - `odds` es una serie cuyo índice son los nombres de las variables: `odds["cat__tarifa_no_reembolsable"]` saca el valor de una de ellas.
    - `odds.idxmax()` devuelve el **nombre** de la variable con el mayor valor.
- **Qué debe salir:**
    - `✔ Ejercicio 2 correcto`. `or_no_reembolsable` ≈ 0,44, `or_deposito_total` ≈ 0,46 y `mas_riesgo` vale `'num__antelacion_dias'`.
    - Las dos líneas con los odds ratio.
- **Si falla:**
    - `La tarifa no reembolsable reduce el riesgo...` → has sacado otra variable o el nombre no es exacto (`cat__tarifa_no_reembolsable`).
    - `Un depósito total reduce el riesgo...` → revisa el nombre `cat__deposito_total`.
    - `assert isinstance(mas_riesgo, str)...` → `idxmax()` devuelve el nombre; con `max()` saldría el número.
    - `KeyError` → el nombre de la variable está mal escrito (llevan el prefijo `cat__` o `num__`).

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuánto valen `or_no_reembolsable` y `or_deposito_total`?* → **≈ 0,44** y **≈ 0,46**.
2. *Interpreta el 0,44 de la tarifa no reembolsable con tus palabras.* → Los odds de cancelar se **multiplican por 0,44** (se reducen un 56 %) frente a una tarifa flexible: con tarifa no reembolsable se cancela mucho menos.
3. *¿Qué variable aumenta más el riesgo y cuánto?* → **`num__antelacion_dias`**: odds ×2,12 por cada desviación típica de antelación.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
or_no_reembolsable = odds["cat__tarifa_no_reembolsable"]
or_deposito_total = odds["cat__deposito_total"]
mas_riesgo = odds.idxmax()
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

## 4. El umbral y la matriz de confusión

#### A6 · Ejemplo — Matriz de confusión con dos umbrales

El modelo da una probabilidad; **nosotros decidimos** a partir de qué valor lo llamamos «riesgo de cancelación». Esa decisión es el **umbral**.

La **matriz de confusión** cuenta los cuatro resultados posibles (con «cancela» = positivo):

| | Predicho: se mantiene | Predicho: cancela |
|---|---|---|
| **Real: se mantiene** | Verdadero negativo (VN) | **Falsa alarma** (FP) |
| **Real: cancela** | **Cancelación no prevista** (FN) | Verdadero positivo (VP) |

- **Precisión** = VP / (VP + FP): de las alarmas, ¿cuántas eran ciertas?
- **Recall** = VP / (VP + FN): de las cancelaciones reales, ¿cuántas avisamos?

- **Qué hace:**
    - `metricas_umbral(y_real, proba, umbral)` es una función: convierte las probabilidades en clases (`proba >= umbral`), cuenta VN, FP, FN y VP y calcula precisión y recall. Queda guardada para usarla después.
    - El bucle dibuja dos matrices de confusión de test, una con umbral 0,5 y otra con umbral 0,3 (`ConfusionMatrixDisplay`).
- **Qué debe salir:**
    - Dos matrices. Umbral 0,5: VN 472 · FP 177 · FN 212 · VP 339. Umbral 0,3: VN 262 · FP 387 · FN 69 · VP 482.
- **Si falla:**
    - `NameError: name 'proba_test' is not defined` → falta el ejercicio 1.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *Con umbral 0,5, ¿cuántas cancelaciones no prevé el modelo (FN) y cuántas falsas alarmas da (FP)?* → **212** cancelaciones no previstas y **177** falsas alarmas.
2. *¿Y con umbral 0,3?* → **69** no previstas y **387** falsas alarmas.
3. *¿Qué ocurre al bajar el umbral de 0,5 a 0,3?* → Se marcan **más reservas como riesgo**: las cancelaciones no previstas bajan (212 → 69) pero las falsas alarmas suben (177 → 387). No hay umbral que reduzca los dos errores a la vez.

</details>

#### A7 · Ejercicio 3 — Efecto de bajar el umbral

- **Qué hace:**
    - `metricas_umbral(y_test, proba_test, 0.5)` y `(…, 0.3)` devuelven un diccionario con VN, FP, FN, VP, precisión y recall de cada umbral.
    - `sube_recall = m03["recall"] - m05["recall"]` y `baja_precision = m05["precision"] - m03["precision"]` miden lo que se gana y lo que se pierde al bajar el umbral.
- **Qué debe salir:**
    - `Con umbral 0,3 detectas 26 puntos más de las cancelaciones, pero tu precisión baja 10 puntos.`
    - `✔ Ejercicio 3 correcto`.
- **Si falla:**
    - `La matriz debe sumar todas las reservas de test` → no has llamado a `metricas_umbral` con `y_test` y `proba_test`.
    - `Bajar el umbral debe subir el recall` → la resta está al revés: `m03` menos `m05`.
    - `Bajar el umbral suele bajar la precisión` → la resta de precisión está al revés: `m05` menos `m03`.
    - `NameError: name 'metricas_umbral' is not defined` → falta ejecutar el ejemplo de la matriz de confusión.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuánto valen recall y precisión con umbral 0,5 y con 0,3?* → Umbral 0,5: recall **0,615** y precisión **0,657**. Umbral 0,3: recall **0,875** y precisión **0,555**.
2. *¿Qué significa «detectas 26 puntos más»?* → Que con 0,3 se avisa de **el 87,5 %** de las cancelaciones reales (482 de 551) frente al 61,5 % (339 de 551): 26 puntos más.
3. *¿Qué se paga a cambio?* → La precisión baja 10 puntos (del 65,7 % al 55,5 %): de cada 100 alarmas, **unas 45 son falsas** (antes, unas 34). Si cada falsa alarma tiene un coste, hay que compararlo con lo que se gana (siguiente sección).

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
m05 = metricas_umbral(y_test, proba_test, 0.5)
m03 = metricas_umbral(y_test, proba_test, 0.3)
sube_recall = m03["recall"] - m05["recall"]
baja_precision = m05["precision"] - m03["precision"]
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

## 5. Elegir el umbral según el coste

#### A8 · Ejemplo — Probabilidades de validación cruzada y costes de los errores

**Idea.** Ambos errores no cuestan lo mismo:
- **Cancelación no prevista (FN):** habitación vacía que podíamos haber vendido.
- **Falsa alarma (FP):** molestamos a un cliente que iba a venir, o hacemos *overbooking* sin necesidad.

Si un FN cuesta el **triple** que un FP, conviene ser más «alarmista» (umbral bajo). La regla teórica para el umbral óptimo es `coste_FP / (coste_FP + coste_FN)`; aquí lo comprobamos **empíricamente**. El umbral se elige con probabilidades **de validación cruzada sobre train** (nunca con el test, que se reserva para la evaluación final).

- **Qué hace:**
    - `cross_val_predict(logit, X_train, y_train, cv=cv, method="predict_proba")[:, 1]` da, para cada reserva de train, la probabilidad de cancelar calculada por un modelo que **no la ha visto** (validación cruzada de 5 folds).
    - `COSTE_FN, COSTE_FP = 3.0, 1.0` fija el coste relativo de cada error: son **decisiones de negocio**, no tienen valor por defecto.
    - `coste_total(...)` es una función que calcula el coste de un umbral: `FN × coste_FN + FP × coste_FP`.
- **Qué debe salir:**
    - `Probabilidades de validación cruzada: (4800,) · coste FN: 3.0 · coste FP: 1.0`.
- **Si falla:**
    - `NameError: name 'logit' is not defined` → falta el ejercicio 1.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuántas probabilidades hay en `proba_cv` y por qué son tantas?* → **4800**: una por cada reserva de **train**, cada una calculada con un modelo que no la vio.
2. *¿Por qué `proba_cv` y no `proba_test` para elegir el umbral?* → Porque el test se reserva para la **evaluación final**: si eligiéramos el umbral mirándolo, ya no sería un examen sorpresa.
3. *Con FN = 3 y FP = 1, ¿qué umbral predice la regla teórica `coste_FP / (coste_FP + coste_FN)`?* → **1 / (1 + 3) = 0,25**: un umbral bajo, porque dejar escapar una cancelación sale más caro.

</details>

#### A9 · Ejercicio 4 — Elegir el umbral de menor coste

- **Qué hace:**
    - `umbrales = np.arange(0.05, 0.96, 0.05)` crea los 19 umbrales a probar, de 0,05 a 0,95.
    - `costes` guarda el coste total de cada umbral sobre `proba_cv` / `y_train` (validación cruzada en train).
    - `np.argmin(costes)` da la posición del menor coste; `umbrales[...]` la convierte en el umbral (`mejor_umbral`).
    - `coste_test` es el coste de ese umbral sobre test, **una sola vez**.
- **Qué debe salir:**
    - `Umbral elegido: 0.25 · coste en test: 577 (con 0,5: 813)`.
    - `✔ Ejercicio 4 correcto`.
- **Si falla:**
    - `Con FN 3 veces más caro que FP, el umbral óptimo debería ser bajo` → has calculado los costes sobre `y_test` / `proba_test` o con `COSTE_FN` y `COSTE_FP` en otro orden.
    - `El umbral elegido debería costar menos que el 0,5 por defecto` → `coste_test` no está calculado con `mejor_umbral` sobre test.
    - `NameError: name 'proba_cv' is not defined` → falta el ejemplo anterior.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué umbral sale y cuánto cuesta en test frente al 0,5?* → **0,25**, con un coste de **577** frente a **813** del umbral 0,5.
2. *¿Cuánto se ahorra respecto al umbral por defecto?* → 813 − 577 = **236 unidades** de coste (≈ **29 %** menos).
3. *¿Por qué el umbral óptimo es bajo (0,25) y no 0,5?* → Porque un FN cuesta el **triple** que un FP: compensa dar más alarmas (aunque algunas sean falsas) para no perder cancelaciones. Coincide con la regla teórica 1 / (1 + 3) = 0,25.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
umbrales = np.arange(0.05, 0.96, 0.05)
costes = [coste_total(y_train, proba_cv, u, COSTE_FN, COSTE_FP) for u in umbrales]
mejor_umbral = float(umbrales[int(np.argmin(costes))])
coste_test = coste_total(y_test, proba_test, mejor_umbral, COSTE_FN, COSTE_FP)
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

#### A10 · Ejemplo — Curva del coste según el umbral

- **Qué hace:**
    - Dibuja el coste total (`costes`) de cada umbral de `umbrales`, y una línea roja vertical en el umbral de menor coste.
- **Qué debe salir:**
    - Una curva con forma de «U» asimétrica con el mínimo en 0,25 (coste ≈ 2207 en train): baja deprisa hasta ahí y sube mucho hacia la derecha (≈ 6524 con 0,95).
- **Si falla:**
    - `NameError: name 'costes' is not defined` → falta el ejercicio 4 (o su solución).

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Dónde está el mínimo de la curva y cuánto vale aproximadamente?* → En **0,25**, con un coste ≈ **2207** (sobre train, por validación cruzada).
2. *¿Por qué el coste sube tanto hacia la derecha?* → Porque con un umbral alto casi no se avisa de nada: aparecen **muchas cancelaciones no previstas**, y cada una cuesta 3 unidades. Con 0,95 el coste (≈ 6524) es unas 3 veces el mínimo.
3. *¿Qué pasa si se cambia a un FN que cuesta lo mismo que un FP?* → El mínimo se movería hacia **0,5**: sin penalizar más un error que el otro, el umbral «neutro» es el mejor.

</details>

## 6. Mini-reto: tabla para el gerente

#### A11 · Mini-reto — Tabla de umbrales para el gerente

- **Qué hace:**
    - Recorre los tres umbrales (0,5, 0,3 y `mejor_umbral`) y, para cada uno, calcula con `metricas_umbral` el recall y la precisión en test y con `coste_total` el coste.
    - `pd.DataFrame(filas)` los junta en la tabla `tabla_umbrales`, una fila por umbral.
    - `recomendacion` es una frase que propone un umbral y lo justifica con al menos una cifra.
- **Qué debe salir:**
    - `✔ Mini-reto correcto` y la tabla: umbral 0,50 → recall 0,615, precisión 0,657, coste 813 · umbral 0,30 → 0,875, 0,555, 594 · umbral 0,25 → 0,920, 0,533, 577.
- **Si falla:**
    - `Debe haber tres filas` → faltan umbrales en la lista, o `filas` está dentro del bucle.
    - `assert {"umbral", "recall", "precision", "coste"} <= set(tabla_umbrales.columns)` → los nombres de las columnas no coinciden.
    - `La recomendación debe ser texto con al menos una cifra` → `recomendacion` no es un texto, o no incluye ningún número.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué umbral tiene el menor coste y cuál es su recall y su precisión?* → **0,25**: coste 577, recall **0,920** y precisión **0,533**.
2. *Explica con una frase para el gerente qué significa precisión 0,533.* → Que **algo más de la mitad de las alarmas son ciertas** (53 %): casi la mitad (47 %) son falsas alarmas, que hay que gestionar sin molestar al cliente.
3. *¿Por qué conviene enseñar al gerente recall, precisión y coste, y no solo el coste?* → Porque el coste depende de **sus propios pesos** (3 y 1): si cambia de opinión, la tabla le permite ver qué detecta y qué falsas alarmas genera con cada umbral.

</details>

## 7. Para reflexionar

1. Un modelo con exactitud 0,67 ¿es bueno para TurisData? ¿Qué preguntas harías antes de contestar?
2. El *odds ratio* de la tarifa no reembolsable es < 1. ¿Significa que **causa** menos cancelaciones? ¿Qué otra explicación hay?
3. ¿Qué pasa con la precisión y el recall si subes el umbral a 0,8?
4. ¿Quién debería fijar los costes de FN y FP: el equipo de datos o el gerente? ¿Por qué?

<details><summary>Ver respuestas (después de pensarlo)</summary>

1. No se puede saber solo con la exactitud. Preguntas: ¿cuál es el *baseline* (aquí, 0,54 con «nunca cancela»)?, ¿cuántas cancelaciones detecta (con umbral 0,5, solo el 61,5 %) y cuántas falsas alarmas da?, ¿cuánto cuesta cada error?, ¿y en validación cruzada o en el futuro (partición temporal)?
2. **No necesariamente.** Es una asociación observada: quien elige tarifa no reembolsable puede ser alguien que ya tiene más claro que viajará (**selección**), y no la tarifa lo que cambia su conducta. La penalización económica probablemente influye, pero los datos observacionales no lo demuestran.
3. La **precisión sube** (con 0,8 llega a 0,89) y el **recall se desploma** (≈ 0,20): solo avisamos cuando estamos muy seguros, y dejamos pasar 4 de cada 5 cancelaciones.
4. **El gerente** (el negocio): cuánto cuesta una habitación vacía frente a molestar a un cliente es una decisión de negocio, no estadística. El equipo de datos ayuda a estimar esos costes y enseña cómo cambia el umbral y los resultados al variar los costes.

</details>

## 8. Resumen: qué has aprendido
- La regresión logística devuelve **probabilidades**; el umbral las convierte en decisiones.
- `exp(coeficiente)` = *odds ratio*: explica el efecto de cada variable en el riesgo.
- La matriz de confusión separa dos errores con costes distintos.
- El umbral se elige según el negocio y con validación, no con el test.